# Node18 — 로컬 실행 (Cursor / Mac)

**Colab 아님.** cwd = 이 노트북 폴더(`08_LLM Trend`).

1. **셀 1** 패키지 일괄 설치  
2. **셀 2** `KoChatGPT` 클론 + `./chatgpt` 복사  
3. **셀 4** 소스 패치 (최초 1회)  
4. **셀 5** `.env`에 `HF_TOKEN=...` 저장 후 Hugging Face 로그인  

*(예전에 노트북에 토큰을 넣었다면 Hugging Face에서 토큰 재발급 권장)*  


In [1]:
# [셀 1] 의존성 한 번에
%pip install -q --upgrade pip
%pip install -q transformers accelerate safetensors huggingface_hub datasets loralib trl peft sentencepiece python-dotenv rouge-score nltk pandas numpy tqdm ipywidgets einops scipy scikit-learn

import subprocess, sys
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
except Exception:
    print("(선택) bitsandbytes 생략")
print("셀 1 완료 → 필요 시 커널 재시작")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
셀 1 완료 → 필요 시 커널 재시작


In [ ]:
# # [셀 2] chatgpt 폴더 준비 (로컬)
# from pathlib import Path
# import shutil, subprocess, sys
# ROOT = Path.cwd()
# KO = ROOT / "KoChatGPT"
# SRC = KO / "colossalai_ChatGPT_230319" / "chatgpt"
# DST = ROOT / "chatgpt"
# if not KO.is_dir():
#     subprocess.run(["git", "clone", "https://github.com/airobotlab/KoChatGPT"], cwd=str(ROOT), check=True)
# if not SRC.is_dir():
#     sys.exit("경로 없음: " + str(SRC))
# if DST.is_dir():
#     print("이미 있음:", DST)
# else:
#     shutil.copytree(SRC, DST)
    # print("복사 완료", DST)


In [ ]:
# chatgpt 초기화: import shutil; shutil.rmtree('chatgpt', ignore_errors=True)
pass


In [ ]:
# [셀 4] ColossalAI 제거 패치 (fresh clone 후 1회)
from pathlib import Path

BASE = Path.cwd() / "chatgpt"
if not BASE.is_dir():
    raise SystemExit("먼저 셀 2 실행")
mods = [
    (BASE / "trainer/callbacks/save_checkpoint.py", [
        (2, "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy", "from chatgpt.trainer.strategies import Strategy"),
        (70, "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)", "            only_rank0 = True  # no ColossalAI"),
    ]),
    (BASE / "trainer/strategies/__init__.py", [
        (0, "from .colossalai import ColossalAIStrategy", ""),
        (4, "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']", "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"),
    ]),
    (BASE / "dataset/reward_dataset.py", [(2, "from tqdm import tqdm", "from tqdm.auto import tqdm")]),
    (BASE / "trainer/base.py", [(7, "from tqdm import tqdm", "from tqdm.auto import tqdm")]),
    (BASE / "trainer/rm.py", [(7, "from tqdm import tqdm", "from tqdm.auto import tqdm")]),
]
for fp, changes in mods:
    if not fp.is_file():
        print("skip", fp)
        continue
    lines = open(fp, encoding="utf-8").readlines()
    ok = False
    for idx, old, new in changes:
        if idx < len(lines) and lines[idx].strip() == old:
            lines[idx] = new + "\n"
            ok = True
        elif idx < len(lines):
            print("이미 패치 또는 불일치:", fp.name, "line", idx + 1)
    if ok:
        open(fp, "w", encoding="utf-8").writelines(lines)
        print("OK", fp.name)
print("패치 끝")


In [2]:
# [셀 5] HF 로그인 — 노트북에 토큰 넣지 말고 .env 사용
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
tok = os.getenv("HF_TOKEN")
if not tok:
    print("`.env` 파일에 HF_TOKEN=your_token 추가 후 재실행")
else:
    login(token=tok)
    print("Hugging Face 로그인 OK")


`.env` 파일에 HF_TOKEN=your_token 추가 후 재실행


In [3]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy
import loralib


In [ ]:
print("Torch", torch.__version__)
print("CUDA", torch.version.cuda)
print("transformers", transformers.__version__)
print("GPU", torch.cuda.is_available())
from chatgpt.trainer.strategies import NaiveStrategy


In [ ]:
print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

## 2. Base model 

In [ ]:
# 디바이스 + KakaoBrain KoGPT (revision KoGPT6B-ryan1.5b-float16)
# https://huggingface.co/kakaobrain/kogpt/tree/KoGPT6B-ryan1.5b-float16 (CC BY-NC-ND — HF에서 라이선스 동의 필요)
device = "cuda" if torch.cuda.is_available() else "cpu"
KOGPT_ID, KOGPT_REVISION = "kakaobrain/kogpt", "KoGPT6B-ryan1.5b-float16"
tokenizer = AutoTokenizer.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION,
    bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]", pad_token="", mask_token="[MASK]",
)
if not tokenizer.pad_token or tokenizer.pad_token == "":
    tokenizer.pad_token = tokenizer.eos_token
_dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION, torch_dtype=_dtype, trust_remote_code=True
).to(device)

In [ ]:
# 최대 토큰 길이 확인
tokenizer.model_max_length

In [ ]:
# 설정값은 모델이 몇 번째 순서까지 기억하고 처리할 수 있는지를 결정
model.config.n_positions

In [ ]:
# 이 셀: 토크나이즈 및 생성 실험에 사용할 예시 한국어 문장을 정의합니다.
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

In [ ]:
# 이 셀: 예시 문장을 토큰 단위와 토큰 ID 벡터로 변환해 봅니다.
tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

In [ ]:
# 이 셀: 토큰과 토큰 ID를 표(DataFrame) 형태로 만들어 사람이 보기 좋게 출력합니다.
pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

In [ ]:
# 이 셀: Greedy decoding 방식으로 예시 문장을 이어서 생성해 봅니다.
max_length=128
enc = tokenizer(input_txt, return_tensors="pt")
input_ids = enc["input_ids"].to(device)
attention_mask = enc.get("attention_mask").to(device) if enc.get("attention_mask") is not None else None
output_greedy = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))

In [ ]:
# 이 셀: Beam search(빔 서치) 방식으로 예시 문장을 생성해 보고 Greedy와 비교합니다.
enc = tokenizer(input_txt, return_tensors="pt")
input_ids = enc["input_ids"].to(device)
attention_mask = enc.get("attention_mask").to(device) if enc.get("attention_mask") is not None else None
output_beam = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=max_length, num_beams=10, no_repeat_ngram_size=2,
                             do_sample=False)
print(tokenizer.decode(output_beam[0]))

In [ ]:
# 이 셀: Beam + 샘플링(temperature, top-k)을 함께 사용해 더 다양한 문장을 생성해 봅니다.
output_beam = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, temperature=2.0, top_k=50)
print(tokenizer.decode(output_beam[0]))

In [ ]:
#데이터셋 확인

import json
data_path_1_SFT = 'KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

In [ ]:
# 이 셀: Reward Model(RM) 학습에 사용할 원본 JSONL 데이터를 로드하고 개수/앞부분을 확인합니다.
data_path_2_RM = 'KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

In [ ]:
# 이 셀: PPO 단계에서 사용할 프롬프트 데이터(JSONL)를 로드하고 개수/앞부분을 확인합니다.
data_path_3_PPO = 'KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

3. Supervised Fine-Tuning (백본: `kakaobrain/kogpt` @ `KoGPT6B-ryan1.5b-float16`)

In [ ]:
#SFT

from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging  
import copy 

In [ ]:
# SFT 파운데이션: kakaobrain/kogpt @ KoGPT6B-ryan1.5b-float16
KOGPT_ID, KOGPT_REVISION = "kakaobrain/kogpt", "KoGPT6B-ryan1.5b-float16"
tokenizer = AutoTokenizer.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION,
    bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]", pad_token="", mask_token="[MASK]",
    padding_side="right", model_max_length=512,
)
if not tokenizer.pad_token or tokenizer.pad_token == "":
    tokenizer.pad_token = tokenizer.eos_token
_dt = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION, torch_dtype=_dt, trust_remote_code=True
)
if torch.cuda.is_available():
    model = model.cuda()
print(tokenizer)

In [ ]:

class SFT_dataset(Dataset):
    """Supervised Fine-Tuning에 사용할 데이터셋 클래스.

    - JSONL 형식의 KoChatGPT SFT 데이터를 읽어와서
      `Instruction(명령어) + Response(응답)` 형태의 텍스트로 만든 뒤
      토크나이즈해서 `input_ids`와 `labels`를 생성합니다.
    - `labels`에서 **명령어 부분(token)** 은 -100으로 마스킹해서
      손실(loss)이 **응답 부분만** 계산되도록 합니다.
    """

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        # JSON 안에서 명령어와 응답이 들어있는 key 이름
        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        # SFT용 JSONL 파일 로드
        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        # Instruction / Response 형식의 프롬프트 템플릿
        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        # 1) sources : 명령어(instruction)만 포함된 텍스트 리스트
        sources = []
        for example in list_data_dict:
            # example 안에는 {"prompt": ..., "completion": ...} 구조가 들어 있음
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        # 2) targets : 정답 응답(completion)에 EOS 토큰을 붙인 텍스트 리스트
        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")

        # 3) examples : Instruction + Response 를 하나의 문자열로 이어붙인 리스트
        examples = [s + t for s, t in zip(sources, targets)]

        # source만, source+target 전체를 각각 토크나이즈
        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        # labels는 input_ids를 그대로 복사해서 시작
        labels = copy.deepcopy(input_ids)

        # labels에서 **프롬프트 구간**은 -100으로 마스킹해서 loss에 포함되지 않도록 처리
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        # 나중에 __getitem__에서 사용할 수 있도록 멤버 변수로 저장
        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d" % (len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        """여러 개의 문자열 리스트를 받아서 토크나이즈하고,
        텐서와 각 문장 길이(패딩 제외)를 반환합니다.
        """
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",        # 가장 긴 문장에 맞춰 패딩
                max_length=tokenizer.model_max_length,
                truncation=True,           # 너무 긴 문장은 잘라냄
            )
            for text in strings
        ]
        # 각 예시의 input_ids 텐서를 뽑아서 리스트로 저장
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        # pad 토큰이 아닌 실제 토큰 개수를 길이로 사용
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        # 데이터셋의 전체 샘플 개수
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        # i번째 샘플의 input_ids / labels를 딕셔너리 형태로 반환
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [ ]:
@dataclass
class DataCollatorForSupervisedDataset(object):
    """SFT용 배치(collate) 함수.

    - `Dataset`에서 꺼낸 여러 개의 샘플을 받아서
      하나의 미니배치 텐서로 합쳐 줍니다.
    - 길이가 서로 다른 문장들을 `pad_sequence`로 맞추고,
      `attention_mask`도 함께 만들어서 모델에 입력할 수 있는 형태로 반환합니다.
    """

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        # instances : [{"input_ids": tensor(...), "labels": tensor(...)}, ...] 형태의 리스트
        # 아래 한 줄로, 각 key에 대해 배치 리스트를 추출합니다.
        input_ids, labels = tuple(
            [instance[key] for instance in instances] for key in ("input_ids", "labels")
        )

        # input_ids 텐서들을 오른쪽 패딩(pad_token_id)으로 길이를 맞추어 하나의 [batch, seq_len] 텐서로 만듭니다.
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )

        # labels도 같은 길이로 패딩하되, 손실을 계산하지 않을 부분은 -100으로 패딩합니다.
        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )

        # 패딩이 아닌 실제 토큰 위치는 True, 패딩 위치는 False인 attention_mask 생성
        attention_mask = input_ids.ne(self.tokenizer.pad_token_id)

        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=attention_mask,
        )

In [ ]:
# 이 셀: SFT용 Dataset/Collator 인스턴스를 만들고, 첫 번째 샘플의 입력/라벨을 확인합니다.
train_dataset = SFT_dataset(data_path_1_SFT='KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

In [ ]:
# train_dataset.input_ids[0]를 디코딩해보세요.

In [ ]:
# 이 셀: SFT 학습에 사용할 TrainingArguments와 Trainer를 설정합니다.
training_args = transformers.TrainingArguments(
    output_dir="test",
    # overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16 = False,
    bf16= False,
    )
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

1. 배치 사이즈 줄이기 
2. gradietn_accumulation_step
3. dataloader_num_workers =4,
4. dataloader_pin_memoery=ㅆ겨ㅜㄷ 

In [ ]:
# # 진행 불가 
# # 이 셀: SFT 학습을 실제로 실행하고, 학습이 끝난 모델을 디스크에 저장합니다.
# trainer.train()
# model.save_pretrained('models/output_1_SFT')

In [ ]:
# 이 셀: SFT가 끝난 모델로 파이프라인을 만들고, 몇 가지 질의에 대해 생성 결과를 확인합니다.
generator = transformers.pipeline('text-generation', model='models/output_1_SFT', tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

In [ ]:
# 이 셀: GPU 메모리를 비워 다음 단계(RM, PPO) 학습 시 메모리 부족을 방지합니다.
torch.cuda.empty_cache()

## 3-1. 정량 평가: BLEU / ROUGE (KoGPT 베이스 vs SFT)

- **BLEU_ROGUE.ipynb**와 동일: 생성문 vs `completion` 기준 BLEU + ROUGE-1/2/L
- **베이스**: `kakaobrain/kogpt` @ `KoGPT6B-ryan1.5b-float16`
- **SFT**: `models/output_1_SFT` (전체 가중치 저장 가정)
- `pip install rouge_score nltk` · VRAM 부족 시 `NUM_SAMPLES` 줄이기 · 베이스 평가 후 `del` + `empty_cache` 뒤 SFT 로드


In [ ]:
# BLEU / ROUGE + 평가용 데이터
# !pip install rouge_score nltk

import json
import torch
from tqdm.auto import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

KOGPT_ID = "kakaobrain/kogpt"
KOGPT_REVISION = "KoGPT6B-ryan1.5b-float16"
SFT_DIR = "models/output_1_SFT"

def calculate_metrics(model, tokenizer, dataset, generation_args, prompt_dict, num_samples=50, num_print_samples=2):
    device = next(model.parameters()).device
    model.eval()
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    smoothie = SmoothingFunction().method4
    samples = dataset[: min(num_samples, len(dataset))]
    n = len(samples)
    if n == 0:
        return (0.0, 0.0, 0.0, 0.0)
    gen_args = dict(generation_args)
    gen_args.setdefault("eos_token_id", tokenizer.eos_token_id)
    if tokenizer.pad_token_id is not None:
        gen_args.setdefault("pad_token_id", tokenizer.pad_token_id)
    tb = tr1 = tr2 = trL = 0.0
    for i, ex in enumerate(tqdm(samples, desc="BLEU/ROUGE")):
        ref = ex["completion"]
        formatted = prompt_dict["prompt_input"].format_map({"prompt": ex["prompt"]})
        enc = tokenizer(formatted, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc.get("attention_mask").to(device) if enc.get("attention_mask") is not None else None
        with torch.no_grad():
            if attention_mask is not None:
                out = model.generate(input_ids=input_ids, attention_mask=attention_mask, **gen_args)
            else:
                out = model.generate(input_ids=input_ids, **gen_args)
        full = tokenizer.decode(out[0], skip_special_tokens=True)
        if "### Response(응답):" in full:
            pred = full.split("### Response(응답):")[-1].strip()
        else:
            pred = full.replace(formatted, "").strip()
        if i < num_print_samples:
            print(f"\n[{i+1}] Q: {ex['prompt'][:60]}...\n  Ref: {ref[:80]}...\n  Pred: {pred[:80]}...")
        tb += sentence_bleu([ref.split()], pred.split(), smoothing_function=smoothie)
        r = scorer.score(ref, pred)
        tr1 += r["rouge1"].fmeasure
        tr2 += r["rouge2"].fmeasure
        trL += r["rougeL"].fmeasure
    avg = (tb / n, tr1 / n, tr2 / n, trL / n)
    print(f"\n[n={n}] BLEU={avg[0]:.4f}  R1={avg[1]:.4f}  R2={avg[2]:.4f}  RL={avg[3]:.4f}")
    return avg

eval_prompt_dict = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}
eval_generation_args = dict(
    num_beams=4, repetition_penalty=2.0, no_repeat_ngram_size=4,
    max_new_tokens=64, do_sample=True, top_k=50, early_stopping=True,
)

with open("KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl", "r", encoding="utf-8-sig") as f:
    sft_eval_list = json.load(f)
print("평가 샘플 수(전체):", len(sft_eval_list))



In [ ]:
# KoGPT 사전학습 vs SFT 체크포인트 비교 (GPU)
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "GPU 권장"
NUM_SAMPLES = 50

tok = AutoTokenizer.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION,
    bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]", pad_token="", mask_token="[MASK]",
)
if not tok.pad_token or tok.pad_token == "":
    tok.pad_token = tok.eos_token
eval_generation_args["eos_token_id"] = tok.eos_token_id
eval_generation_args["pad_token_id"] = tok.pad_token_id

# --- 베이스 ---
base_m = AutoModelForCausalLM.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION, torch_dtype=torch.float16, trust_remote_code=True
).cuda()
base_metrics = calculate_metrics(base_m, tok, sft_eval_list, eval_generation_args, eval_prompt_dict, num_samples=NUM_SAMPLES)
del base_m
torch.cuda.empty_cache()

# --- SFT (저장된 전체 모델) ---
sft_m = AutoModelForCausalLM.from_pretrained(SFT_DIR, torch_dtype=torch.float16, trust_remote_code=True).cuda()
sft_metrics = calculate_metrics(sft_m, tok, sft_eval_list, eval_generation_args, eval_prompt_dict, num_samples=NUM_SAMPLES)
del sft_m
torch.cuda.empty_cache()

print("\n" + "=" * 52)
print(f"{'Metric':<14} | {'Base (pretrain)':<16} | {'SFT':<12}")
print("-" * 52)
for name, a, b in zip(["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L"], base_metrics, sft_metrics):
    print(f"{name:<14} | {a:<16.4f} | {b:<12.4f}")



4. Reward Model

In [ ]:
# RLHF 2단계: Reward Model (KoGPT / GPT-J 백본)

from typing import Optional
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from transformers import GPTJConfig, GPTJModel
import torch.nn as nn
import random


In [ ]:
class GPTJRM_custom(RewardModel):
    """KoGPT (GPT-J) 백본 + value head."""
    def __init__(self, pretrained=None, revision=None, config=None, checkpoint=False, lora_rank=0, lora_train_bias="none", tokenizer=None):
        if pretrained is not None:
            kwargs = dict(trust_remote_code=True, torch_dtype=torch.float16)
            if revision:
                kwargs["revision"] = revision
            model = GPTJModel.from_pretrained(pretrained, **kwargs)
            if tokenizer is not None:
                model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPTJModel(config)
        else:
            model = GPTJModel(GPTJConfig())
        if checkpoint:
            model.gradient_checkpointing_enable()
        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)
        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained
    def save_pretrained(self, dir):
        if getattr(self, "pretrained", None):
            self.model.save_pretrained(dir)


In [ ]:
# RM 초기화: KoGPT6B-ryan1.5b-float16 (GPT-J)
from transformers import AutoTokenizer
KOGPT_ID, KOGPT_REVISION = "kakaobrain/kogpt", "KoGPT6B-ryan1.5b-float16"
tokenizer = AutoTokenizer.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION,
    bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]", pad_token="", mask_token="[MASK]",
)
if not tokenizer.pad_token or tokenizer.pad_token == "":
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = 512
with NaiveStrategy().model_init_context():
    model = GPTJRM_custom(pretrained=KOGPT_ID, revision=KOGPT_REVISION, lora_rank=0, tokenizer=tokenizer, checkpoint=True).cuda()


In [ ]:
# RM 학습용 원본 JSONL (ranking 정보가 포함된 데이터) 로드
with open('KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

# ranking 형식의 데이터를 (prompt, chosen, rejected) 쌍들의 리스트로 변환할 리스트
# 예를 들어 ranking 3개가 있으면, 가능한 조합 3쌍(0 vs 1, 0 vs 2, 1 vs 2)을 만듭니다.
total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    # 1) completion_0 vs completion_1
    data = {}
    data['prompt'] = tmp['prompt']
    # ranking 값이 작을수록 더 좋은(선호되는) 응답이라고 가정
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']   # 더 선호되는 응답
        data['rejected'] = tmp['completion_1'] # 덜 선호되는 응답
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    # 2) completion_0 vs completion_2
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    # 3) completion_1 vs completion_2
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)

    # 한 샘플에서 만든 3쌍을 전체 리스트에 합치기
    total_data_ranking2chosen.extend(one_data_ranking2chosen)

# 변환 전/후 데이터 개수와 예시를 출력해서 확인
print('before data num: %d' % (len(list_data_dict)))
print('after  data num: %d' % (len(total_data_ranking2chosen)))
print('data example: \n%s' % total_data_ranking2chosen[45])

In [ ]:
# class PairWiseLoss(nn.Module):

#     def forward(self, chosen_reward: torch.Tensor, reject_reward: torch.Tensor) -> torch.Tensor:
#         probs = torch.sigmoid(chosen_reward - reject_reward)
#         log_probs = torch.log(probs)
#         loss = -log_probs.mean()
#         return loss

In [ ]:
# 이 셀: RM 학습용 (prompt, chosen, rejected) 데이터 리스트를 섞고, 예시 한 건을 출력해 확인합니다.
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

In [ ]:
# 이 셀: RM 데이터를 train/eval로 나누고, RewardDataset 형태로 감쌉니다.
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

In [ ]:
# 이 셀: RM 학습 데이터 한 건의 prompt / chosen / rejected 내용을 눈으로 확인합니다.
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

In [ ]:
# 이 셀: Reward Model 학습을 수행할 Trainer를 설정합니다.
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=1)

In [ ]:
# 이 셀: Reward Model을 실제로 학습시키고, 학습이 끝난 모델을 저장합니다.
trainer.fit(use_lora=0)

model.save_pretrained('models/output_2_RM')

In [ ]:
# 이 셀: 학습된 Reward Model에 문장을 넣어 reward 점수를 확인하는 함수를 정의하고, 예시 1개를 테스트합니다.
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

In [ ]:
# 이 셀: 정보성/중립적인 문장에 대해 Reward Model 점수를 확인해 봅니다 (예시 2).
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

In [ ]:
# 이 셀: 길이가 더 긴 설명형 문장에 대해 Reward Model 점수를 확인합니다 (예시 3).
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."

output_reward = inference_RM(input_text=input_text)

In [ ]:
# 이 셀: 또 다른 설명 문장에 대해 Reward Model이 얼마나 긍정적으로 평가하는지 확인합니다 (예시 4).
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

In [ ]:
# 이 셀: PPO 학습 전에 다시 한 번 GPU 메모리를 정리합니다.
torch.cuda.empty_cache()

 # 5. Proximal Policy Optimization
 RLHF의 마지막 세번째 단계인 Proximal Policy Optimization(PPO)

In [ ]:
# 이 셀: PPO에 사용할 Actor/ 크리틱 모듈과 PPOTrainer, deepcopy 유틸을 임포트합니다.
# KoGPT(GPT-J) Actor/Critic — NeoX 대체
from chatgpt.models.base import Actor, Critic
from transformers import GPTJConfig, GPTJForCausalLM, GPTJModel

class GPTJActor(Actor):
    def __init__(self, pretrained=None, revision=None, checkpoint=False, lora_rank=0, lora_train_bias="none"):
        kw = dict(trust_remote_code=True, torch_dtype=torch.float16)
        if revision:
            kw["revision"] = revision
        model = GPTJForCausalLM.from_pretrained(pretrained, **kw) if pretrained else GPTJForCausalLM(GPTJConfig())
        if checkpoint:
            model.gradient_checkpointing_enable()
        super().__init__(model, lora_rank, lora_train_bias)

class GPTJCritic(Critic):
    def __init__(self, pretrained=None, revision=None, checkpoint=False, lora_rank=0, lora_train_bias="none", **kwargs):
        kw = dict(trust_remote_code=True, torch_dtype=torch.float16)
        if revision:
            kw["revision"] = revision
        model = GPTJModel.from_pretrained(pretrained, **kw) if pretrained else GPTJModel(GPTJConfig())
        if checkpoint:
            model.gradient_checkpointing_enable()
        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias, **kwargs)

# 아래 PPO에서 사용
NeoXActor = GPTJActor
NeoXCritic = GPTJCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [ ]:
# NaiveStrategy의 model_init_context 안에서 PPO에 사용할 모델들을 초기화합니다.
with NaiveStrategy().model_init_context():
    # 1) Actor : SFT 단계에서 학습한 정책 모델을 불러옵니다.
    actor = GPTJActor(
        pretrained='models/output_1_SFT',
        lora_rank=0
    ).to(torch.cuda.current_device())

    critic = GPTJCritic(
        pretrained='models/output_2_RM',
        lora_rank=0
    ).to(torch.cuda.current_device())

    tokenizer = AutoTokenizer.from_pretrained(
        "kakaobrain/kogpt", revision="KoGPT6B-ryan1.5b-float16",
        bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]", pad_token="", mask_token="[MASK]",
        padding_side="right", model_max_length=512,
    )
    if not tokenizer.pad_token or tokenizer.pad_token == "":
        tokenizer.pad_token = tokenizer.eos_token

    # 4) 초기 정책(초기 actor)을 따로 복사해 둡니다.
    #    PPO에서는 현재 정책과 초기 정책의 차이를 KL 등으로 regularization하는 데 사용할 수 있습니다.
    initial_model = deepcopy(actor)

    # 5) RewardModel : critic의 backbone과 value_head를 이용해 PPO에서 사용할 reward model을 구성
    reward_model = RewardModel(
        deepcopy(critic.model),
        deepcopy(critic.value_head),
    ).to(torch.cuda.current_device())

In [ ]:
# 이 셀: PPO 학습에서 사용할 actor/critic 옵티마이저(Adam)를 정의합니다.
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [ ]:
# 이 셀: NaiveStrategy를 통해 actor/critic, reward_model, initial_model을 분산 전략에 맞게 준비합니다.
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [ ]:
# PPO 학습에 사용할 프롬프트 데이터 로드
with open('KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    # 각 샘플에서 prompt만 추출해서 리스트로 만듭니다.
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]


def tokenize_fn(texts):
    """PPOTrainer에서 사용할 토크나이즈 함수.

    - 문자열(또는 문자열 리스트)을 받아서 KoGPT2 토크나이저로 토크나이즈하고,
      GPU에 올려서 모델이 바로 사용할 수 있는 형태로 반환합니다.
    """
    batch = tokenizer(
        texts,
        return_tensors='pt',
        max_length=96,    # 프롬프트 길이를 96 토큰으로 제한
        padding=True,     # 가장 긴 문장에 맞춰 패딩
        truncation=True,  # 너무 긴 경우 잘라내기
    )
    # input_ids, attention_mask 등을 모두 cuda()로 옮겨서 반환
    return {k: v.cuda() for k, v in batch.items()}

In [ ]:
# 이 셀: tokenize_fn이 제대로 동작하는지 예시 문장 하나로 테스트합니다.
print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

In [ ]:
# 이 셀: PPO 학습에 사용할 프롬프트 개수를 확인합니다.
len(list_prompt)

In [ ]:
# PPO 알고리즘을 실제로 수행할 Trainer 구성
trainer = PPOTrainer(
    NaiveStrategy(),   # 분산 전략(여기서는 단일 GPU / Naive)
    actor,             # 정책(Policy) 모델
    critic,            # 가치(Value) 모델
    reward_model,      # 보상(Reward) 모델
    initial_model,     # 초기 정책(regularization 등에 사용)
    actor_optim,       # actor용 옵티마이저
    critic_optim,      # critic용 옵티마이저
    max_epochs=1,      # PPO 학습 epoch 수 (데모용으로 1 epoch)
    train_batch_size=8,# 한 번에 학습할 배치 크기
    tokenizer=tokenize_fn, # 위에서 정의한 토크나이즈 함수
    max_length=128,    # 에피소드 생성 시 최대 토큰 길이
    do_sample=True,    # 샘플링을 사용할지 여부 (True면 더 다양한 응답 생성)
    temperature=1.0,   # 샘플링 온도 (높을수록 다양성↑, 낮을수록 보수적)
    top_k=50,          # 상위 k개 토큰만 샘플링에 사용
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [ ]:
# 이 셀: PPO 알고리즘으로 정책(actor)을 학습시키고, 학습된 최종 모델을 저장합니다.
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO')

In [ ]:
def generation(input_text, model):
    """학습된 모델로부터 응답을 생성하고 출력하는 함수.

    - `input_text` : Instruction(명령어) + 기타 프롬프트 텍스트
    - `model`      : PPO까지 학습이 완료된 actor 모델
    """
    # 입력 텍스트를 토큰 ID로 변환하고 GPU에 올립니다.
    enc = tokenizer(
        input_text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    )
    input_ids = enc['input_ids'].to(torch.cuda.current_device())
    attention_mask = enc.get('attention_mask')
    if attention_mask is not None:
        attention_mask = attention_mask.to(torch.cuda.current_device())

    # generate 함수로 최대 250 토큰까지 응답을 생성합니다.
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=250,
        do_sample=True,   # 샘플링 기반 생성 (다양한 문장 생성)
        top_k=50,         # 상위 50개 토큰만 후보로 사용
        top_p=0.95,       # 상위 누적 확률 0.95에 해당하는 토큰들만 사용
        num_return_sequences=1,
    )

    # 생성된 토큰 시퀀스를 사람이 읽을 수 있는 문자열로 디코딩
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    print()
    print(output)
    return output

# Instruction / Response 형식의 프롬프트 템플릿
PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

# 실제로 테스트해 볼 질의들
list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?',
]

# 위 질의들을 Instruction/Response 포맷에 맞게 변환
list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

# PPO까지 학습된 actor 모델에 질의들을 넣어보고 결과를 확인
for input_text in list_prompt:
    output = generation(input_text, actor)

6. 미므리 

In [ ]:
# 이 셀: 기존 SFT 데이터(kochatgpt_1_SFT.jsonl)와 새 SFT 데이터(new_kochatgpt_1_SFT.jsonl)를
# 합쳐서 하나의 리스트 merged_sft_data로 만드는 스켈레톤 코드입니다.
# 실제 웹 크롤링/전처리 코드는 외부에서 수행했다고 가정하고,
# 여기서는 두 JSONL 파일이 준비되어 있을 때 어떻게 합칠지만 예시로 보여줍니다.

import json
from pathlib import Path

# 기존 KoChatGPT SFT 데이터 경로
orig_sft_path = Path("KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl")

# 새로 구축한 SFT 데이터 (예: 웹 크롤링/정제 후 만든 파일)
new_sft_path = Path("KoChatGPT/data_kochatgpt/new_kochatgpt_1_SFT.jsonl")

orig_sft_data = []
new_sft_data = []

if orig_sft_path.exists():
    with orig_sft_path.open("r", encoding="utf-8-sig") as f:
        orig_sft_data = json.load(f)
else:
    print("경고: 기존 SFT 파일을 찾을 수 없습니다:", orig_sft_path)

if new_sft_path.exists():
    with new_sft_path.open("r", encoding="utf-8-sig") as f:
        new_sft_data = json.load(f)
else:
    print("알림: 신규 SFT 파일이 아직 없습니다 (선택 사항):", new_sft_path)

print(f"기존 SFT 샘플 수: {len(orig_sft_data)}")
print(f"신규 SFT 샘플 수: {len(new_sft_data)}")

# 두 데이터셋을 단순 결합 (중복 제거/필터링 로직은 필요에 따라 추가 가능)
merged_sft_data = orig_sft_data + new_sft_data
print(f"결합 후 전체 SFT 샘플 수: {len(merged_sft_data)}")

# 이후에는 SFT_dataset를 파일 기반이 아닌 리스트 기반으로 정의해
# merged_sft_data를 직접 넘겨주는 방식으로 학습에 사용할 수 있습니다.
# (기존 SFT_dataset는 파일 경로를 받으므로, 새 클래스를 추가하는 것이 안전합니다.)

## 섹션 1 (재설정): 로컬 환경에서 안전하게 SFT 학습하기

아래 두 코드 셀은 **기존 코드 변경 없이** 로컬 Mac 환경에서 SFT를 다시 시도하기 위한
디바이스 설정과 `TrainingArguments` 재설정을 제공합니다.
(이 섹션의 셀들을 위에서 아래로 다시 실행한 뒤, 기존 `trainer.train()` 셀을 실행하면 됩니다.)

In [ ]:
# 이 셀: 섹션 1 재설정용으로, 디바이스를 CPU로 고정하고 KoGPT2 base 모델/토크나이저를 다시 로드합니다.
# (기존 "cuda if available" 코드는 위쪽에 그대로 남겨두고, 여기서만 안전한 설정으로 덮어씁니다.)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# GPU/MPS OOM을 피하기 위해 학습은 CPU에서만 진행
device = "cpu"

KOGPT_ID, KOGPT_REVISION = "kakaobrain/kogpt", "KoGPT6B-ryan1.5b-float16"
tokenizer = AutoTokenizer.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION,
    bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]", pad_token="", mask_token="[MASK]",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    KOGPT_ID, revision=KOGPT_REVISION, torch_dtype=torch.float32, trust_remote_code=True
).to(device)

print("섹션1 재설정: device =", device)

In [ ]:
# 이 셀: 섹션 1 재설정용 SFT 학습 설정입니다.
# - 배치 크기를 2로 줄이고
# - fp16/bf16 을 끄고
# - tqdm 진행바/로그를 활성화해
# 로컬 Mac 환경에서도 돌아가도록 조정했습니다.

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="test",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16=False,
    bf16=False,
    disable_tqdm=False,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("섹션1 재설정: TrainingArguments 및 Trainer가 준비되었습니다.")

## 섹션 2 (재정리): SFT 데이터 EDA 및 정제

이 섹션에서는 기존 SFT 데이터(`kochatgpt_1_SFT.jsonl`)를 간단히 살펴보고(EDA),
너무 짧거나 이상한 샘플을 필터링한 **정제 버전 SFT 데이터 리스트**를 만드는 코드를 추가합니다.
기존 `SFT_dataset` 클래스와 학습 코드는 그대로 두고, 여기에서는 분석/정제만 별도로 진행합니다.

In [ ]:
# 이 셀: SFT 원본 데이터(`kochatgpt_1_SFT.jsonl`)를 로드해서 길이/예시 등을 간단히 EDA합니다.

import json
from pathlib import Path

sft_path = Path("KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl")

if not sft_path.exists():
    print("경고: SFT 파일을 찾을 수 없습니다:", sft_path)
else:
    with sft_path.open("r", encoding="utf-8-sig") as f:
        sft_data = json.load(f)

    print("SFT 샘플 수:", len(sft_data))
    print("\n[예시 3건]")
    for ex in sft_data[:3]:
        print("- prompt:", ex.get("prompt", ""))
        print("  completion:", ex.get("completion", ""))
        print("  prompt_len:", len(ex.get("prompt", "")), "completion_len:", len(ex.get("completion", "")))
        print("---")

In [ ]:
# 이 셀: 간단한 규칙 기반으로 SFT 데이터를 정제(clean)하는 함수를 정의하고 적용합니다.
# - 너무 짧은 prompt / completion 제거
# - 빈 문자열/None 제거

MIN_PROMPT_LEN = 5      # 글자 기준 최소 길이 (예시 값)
MIN_COMPLETION_LEN = 10 # 글자 기준 최소 길이 (예시 값)


def clean_sft_records(records):
    """간단한 길이/비어 있음 기준으로 SFT 레코드를 필터링합니다.

    records: [{"prompt": str, "completion": str, ...}, ...]
    반환: 필터링된 리스트
    """
    cleaned = []
    for r in records:
        prompt = (r.get("prompt") or "").strip()
        completion = (r.get("completion") or "").strip()

        # 너무 짧거나 비어 있는 경우 제외
        if len(prompt) < MIN_PROMPT_LEN:
            continue
        if len(completion) < MIN_COMPLETION_LEN:
            continue

        new_r = dict(r)
        new_r["prompt"] = prompt
        new_r["completion"] = completion
        cleaned.append(new_r)

    return cleaned


if "sft_data" in globals():
    clean_sft_data = clean_sft_records(sft_data)
    print("정제 전 샘플 수:", len(sft_data))
    print("정제 후 샘플 수:", len(clean_sft_data))
else:
    print("먼저 바로 위 EDA 셀을 실행해 sft_data를 로드해 주세요.")

In [ ]:
# 이 셀: 정제된 SFT 리스트(clean_sft_data)를 사용하는 Dataset 클래스를 정의합니다.
# 기존 SFT_dataset는 파일 경로를 받으므로, 여기서는 리스트 기반 버전을 새로 만듭니다.

from torch.utils.data import Dataset
from typing import Sequence, Dict
import copy
import transformers
import logging


class SFTDatasetFromList(Dataset):
    """정제된 SFT 레코드 리스트를 직접 받아 사용하는 Dataset.

    records: [{"prompt": str, "completion": str, ...}, ...]
    tokenizer: transformers.PreTrainedTokenizer
    """

    def __init__(self, records, tokenizer: transformers.PreTrainedTokenizer):
        super().__init__()
        logging.warning("Loading cleaned SFT data from list...")

        pattern_instruction = "prompt"
        pattern_output = "completion"

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }
        prompt_input = PROMPT_DICT["prompt_input"]

        # 1) sources: instruction만 포함된 텍스트
        sources = []
        for ex in records:
            tmp = prompt_input.format_map(ex)
            sources.append(tmp)

        # 2) targets: completion + EOS
        targets = []
        for ex in records:
            targets.append(f"{ex[pattern_output]}{tokenizer.eos_token}")

        # 3) examples: Instruction + Response
        examples = [s + t for s, t in zip(sources, targets)]

        # 토크나이즈
        sources_tokenized = self._tokenize_fn(sources, tokenizer)
        examples_tokenized = self._tokenize_fn(examples, tokenizer)

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)

        # 프롬프트 부분은 -100으로 마스킹 (loss에서 제외)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        self.input_ids = input_ids
        self.labels = labels

        logging.warning("Cleaned SFT data loaded: %d samples" % (len(self.labels)))

    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = [t.input_ids[0] for t in tokenized_list]
        input_ids_lens = [
            t.input_ids.ne(tokenizer.pad_token_id).sum().item() for t in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            input_ids_lens=input_ids_lens,
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return {"input_ids": self.input_ids[i], "labels": self.labels[i]}

In [ ]:
# 이 셀: 정제된 SFT 데이터로 학습용 Dataset/Collator를 준비하고, 간단히 예시를 확인합니다.
# (기존 train_dataset / data_collator는 그대로 두고, 별도의 clean_* 버전을 만듭니다.)

if "clean_sft_data" in globals():
    clean_train_dataset = SFTDatasetFromList(clean_sft_data, tokenizer)
    clean_data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

    print("clean_train_dataset 길이:", len(clean_train_dataset))
    print("예시 input_ids[0]:", clean_train_dataset.input_ids[0])
    print("예시 labels[0]:", clean_train_dataset.labels[0])
else:
    print("먼저 정제 함수 셀을 실행해 clean_sft_data를 생성해 주세요.")